# Sentinel-1 and Sentinel-2 parcel zonal statistics

Configure and run `SatelliteZonalStats`. openEO creates one cloud-masked, indexed monthly NetCDF cube per spatial batch. IQR cleaning, null filling, and parcel statistics run locally, producing one ML-ready GeoParquet row per parcel with dated feature columns.

In [ ]:
from pathlib import Path

from satellites.data_preparation.parcel_stats import estimate_utm_epsg_from_parcels
from satellites.data_preparation.parcel_stats.multiuser import (
    split_geodataframe_by_grid,
    load_openeo_users_from_env,
    run_parallel_extractions,
    merge_and_save_results,
)
from satellites.data_preparation.parcel_stats.job_manager import (
    compute_tile_width,
    compute_tile_buffer_metres,
)
from satellites.shared.io import read_data
import satellites.shared.geometry as geom_l

## Inputs

Processing order:

`Sentinel-2 L2A`
`acquisition cloud mask`
`acquisition pixel indices`
`monthly pixel median`
`monthly per-variable cube IQR outlier removal`
`local null filling`
`parcel statistics`
`one-row-per-parcel time-series pivot`
`ML-ready GeoParquet`

Temporal filling is past-only so later observations cannot leak into earlier feature periods.

In [ ]:
# ============================================================================
# STEP 1: Read data and re-project
# ============================================================================
# Read the data and re-project to the desired EPSG

WORK_PATH = Path("C:/work_dir")
RUN_IDENTIFIER = "volvi-2023-2024"  # Change this for each named run.

PARCELS_PATH = WORK_PATH / RUN_IDENTIFIER / "parcels_volvi.gpkg"

In [ ]:
# WORKING_EPSG is selected from the dissolved parcel centroid after loading the data.

In [ ]:
gdf_parcels = read_data(str(PARCELS_PATH))
gdf_parcels = geom_l.return_valid_geometries(gdf_parcels)
gdf_parcels = geom_l.convert_multipolygons_to_polygons(gdf_parcels)

# Set CRS to WGS84 if missing, then re-project to the working CRS.
if gdf_parcels.crs is None:
    print("WARNING: No CRS found on parcels GeoDataFrame. Assuming EPSG:4326 (WGS84).")
    gdf_parcels = gdf_parcels.set_crs(epsg=4326)

WORKING_EPSG = estimate_utm_epsg_from_parcels(gdf_parcels)
print(f"Selected working CRS from parcel centroid: EPSG:{WORKING_EPSG}")

print(f"Re-projecting parcels to EPSG:{WORKING_EPSG}...")
gdf_parcels = gdf_parcels.to_crs(WORKING_EPSG)

print(len(gdf_parcels), "parcels in study area")
gdf_parcels.head(3)

In [ ]:
# ============================================================================
# Define the class column and parcel ID column names.
# These are used in the extraction process to identify the class of each parcel
# and its unique identifier.
# ============================================================================
class_column = "initial_label"
parcel_id_column = "parcel_code"

In [ ]:
# some basic accuracy stats on the predicted labels as coming from services
all_gdf = gdf_parcels.copy()
all_gdf["accuracy"] = all_gdf["initial_label_code"] == all_gdf["predicted_label_code"]
accuracy = all_gdf["accuracy"].mean()
print(f"Accuracy on all data: {accuracy:.2%}")

train_gdf = gdf_parcels[gdf_parcels["set_type"]=="train"].copy()
print(len(train_gdf), "rows in train_gdf after filtering with set_type=='train'")
train_gdf["accuracy"] = train_gdf["initial_label_code"] == train_gdf["predicted_label_code"]
accuracy = train_gdf["accuracy"].mean()
print(f"Accuracy on train data: {accuracy:.2%}")

test_gdf = gdf_parcels[gdf_parcels["set_type"]=="test"].copy()
print(len(test_gdf), "rows in test_gdf after filtering with set_type=='test'")
test_gdf["accuracy"] = test_gdf["initial_label_code"] == test_gdf["predicted_label_code"]
accuracy = test_gdf["accuracy"].mean()
print(f"Accuracy on test data: {accuracy:.2%}")

## Multi-user extraction planning

Load the users first, create one initial grid partition per user, and then select the largest tile width that still supplies two simultaneous jobs to every active account.

In [ ]:
# ============================================================================
# STEP 2: Define the grid and split the parcels into tiles
# ============================================================================
# Create a grid of tiles over the parcels and split the parcels into tiles. Each tile will be processed separately.

OPENEO_JOBS_PER_USER = 2  # This is default by openEO
users_list = load_openeo_users_from_env()

print(f"Loaded {len(users_list)} openEO user(s):")
for i, (user, _) in enumerate(users_list, 1):
    print(f"  User {i}: {user}")

In [ ]:
# One initial partition per user lets all account queues start immediately.
spatial_parts = split_geodataframe_by_grid(
    gdf=gdf_parcels,
    partition_count=len(users_list),
)

print(f"Created {len(spatial_parts)} spatial partitions:")
for i, part in enumerate(spatial_parts, 1):
    print(f"  Partition {i}: {len(part)} parcels")

In [ ]:
# ============================================================================
# Select a tile width that fills both remote slots in every partition.
# ============================================================================
TILE_SIZE_METRES = compute_tile_width(
    gdf_parcels,
    parcel_id_col=parcel_id_column,
    working_epsg=WORKING_EPSG,
    spatial_parts=spatial_parts,
    user_count=len(users_list),
    jobs_per_user=OPENEO_JOBS_PER_USER,
)
TILE_BUFFER_METRES = compute_tile_buffer_metres(gdf_parcels, WORKING_EPSG)

In [ ]:
TILE_SIZE_METRES = 30_000  # 30 km
TILE_BUFFER_METRES = 2_000 # 2 Km

In [ ]:
# ============================================================================
# Confirm the planned global remote concurrency.
# ============================================================================
TOTAL_REMOTE_JOB_CAPACITY = len(users_list) * OPENEO_JOBS_PER_USER
print(
    f"Remote capacity: {len(users_list)} users x {OPENEO_JOBS_PER_USER} jobs "
    f"= {TOTAL_REMOTE_JOB_CAPACITY} simultaneous jobs"
)


In [ ]:
# ============================================================================
# STEP 3: Create extraction configuration dictionary
# ============================================================================
# Single dictionary containing all parameters for the extraction process,
# to be passed to each parallel worker.

START_DATE = "2023-10-01"
END_DATE = "2024-09-30"
OUTPUT_DIR = WORK_PATH / RUN_IDENTIFIER / "1_parcel_stats"

SENTINEL2_BANDS = ["B02", "B03", "B04", "B05", "B08", "B11", "B12"]
SENTINEL2_INDICES = ["EVI", "NDMI", "NDRE", "NDVI", "NDWI",  "SAVI"]

SENTINEL1_BANDS = ["VV", "VH"]
SENTINEL1_INDICES = ["R", "RVI"]
SENTINEL1_ORBIT_DIRECTION = "ASCENDING"  # ASCENDING, DESCENDING, or BOTH

# Mean is mandatory; percentiles enable derived IQR and robust skewness.
SPATIAL_STATISTICS = [
    "mean",
    "median",
    "sd",
    "min",
    "max",
    "range",
]

INTERPOLATION_METHOD = "nearest"  # "idw"
INTERPOLATION_MAX_DISTANCE_IN_METERS = 500

# IQR outliers become null, then nulls are filled.
REMOVE_OUTLIERS = True
IQR_QUANTILES = (0.1, 0.9)
IQR_MULTIPLIER = 1.5
FILL_NULLS = True
IQR_MIN_VALID_PIXELS = 20
TEMPORAL_FILL_MODE = "bidirectional" # or past_only
MINIMUM_PARCEL_PIXELS = 3
MINIMUM_OBSERVED_FRACTION_FOR_FILL = 0.20

# Each worker loads one complete monthly cube; use one unless RAM has been sized for parallel cubes.
LOCAL_BATCH_WORKERS = 1

extraction_config = {
    # Output paths
    "output_dir": OUTPUT_DIR,
    "run_identifier": RUN_IDENTIFIER,

    # Parcel configuration
    "parcel_id_field": parcel_id_column,

    # Temporal bounds
    "start_date": START_DATE,
    "end_date": END_DATE,

    # Spatial configuration
    "working_epsg": WORKING_EPSG,

    # Sentinel-2 configuration
    "sentinel2_bands": SENTINEL2_BANDS,
    "sentinel2_indices": SENTINEL2_INDICES,

    # Sentinel-1 configuration
    "sentinel1_bands": SENTINEL1_BANDS,
    "sentinel1_indices": SENTINEL1_INDICES,
    "sentinel1_orbit_direction": SENTINEL1_ORBIT_DIRECTION,

    # Statistics to calculate
    "spatial_statistics": SPATIAL_STATISTICS,

    # Tiling configuration
    "tile_size_metres": TILE_SIZE_METRES,
    "tile_buffer_metres": TILE_BUFFER_METRES,

    # Data cleaning (IQR outlier removal)
    "remove_outliers": REMOVE_OUTLIERS,
    "iqr_quantiles": IQR_QUANTILES,
    "iqr_multiplier": IQR_MULTIPLIER,
    "iqr_min_valid_pixels": IQR_MIN_VALID_PIXELS,

    # Null filling
    "fill_nulls": FILL_NULLS,
    "temporal_fill_mode": TEMPORAL_FILL_MODE,
    "minimum_parcel_pixels": MINIMUM_PARCEL_PIXELS,
    "minimum_observed_fraction_for_fill": MINIMUM_OBSERVED_FRACTION_FOR_FILL,

    # Interpolation
    "interpolation_method": INTERPOLATION_METHOD,
    "interpolation_max_distance_in_meters": INTERPOLATION_MAX_DISTANCE_IN_METERS,

    # Parallel processing
    "openeo_parallel_jobs": OPENEO_JOBS_PER_USER,
    "job_poll_seconds": 30,
    "batch_workers": LOCAL_BATCH_WORKERS,

    # Other options
    "spatial_fill_window_sizes": (3, 5),
}

In [ ]:
# ============================================================================
# STEP 4: Run all partitions in parallel
# ============================================================================
# Users run in parallel; each user's assigned partitions run sequentially.
# Each active partition manager can run two openEO jobs simultaneously.
run_parallel_extractions(
    spatial_parts=spatial_parts,
    users_list=users_list,
    config=extraction_config,
)

# Each partition result is saved below OUTPUT_DIR/partition_<n>.
print("\nExtraction complete!")

In [ ]:
# ============================================================================
# STEP 4: Merge results and save to files
# ============================================================================
# Combines all partition results and removes duplicates
# Saves one file:
#   1. satellite_parcel_time_stats.geoparquet (full results with geometry)

merged_gdf, geoparquet_path = merge_and_save_results(
    output_dir=OUTPUT_DIR,
    parcel_id_column=parcel_id_column,
    working_epsg=WORKING_EPSG
)

In [ ]:
# ============================================================================
# STEP 5: Inspect results
# ============================================================================

print(f"\n{'='*70}")
print("FINAL RESULTS")
print(f"{'='*70}")

print("\nMerged GeoDataFrame:")
print(f"  Rows: {len(merged_gdf)}")
print(f"  Columns: {len(merged_gdf.columns)}")
print(f"  Has geometry: {merged_gdf.geometry.name in merged_gdf.columns}")

print("\nOutput files:")
print(f"  GeoParquet: {geoparquet_path}")
print(f"    Size: {geoparquet_path.stat().st_size / 1e6:.1f} MB")

print("\nFirst 5 rows:")
display(merged_gdf.head())